# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR\textsuperscript{2} dataset using the `mlcroissant` library. This includes loading the dataset using its Croissant schema, inspecting available record sets and fields, extracting data, basic analysis, and visualization.

### Dataset Source
The dataset source is specified by a Croissant schema at: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset from the Croissant schema
dataset = mlc.Dataset(croissant_url)

# Show basic information about the dataset
metadata_json = dataset.metadata.to_json()

print('Dataset Name:', getattr(dataset.metadata, 'name', ''))
print('Dataset Description:', getattr(dataset.metadata, 'description', ''))

## 2. Data Overview
Review available record sets, their `@id`s, and fields/columns for further exploration.

In [ ]:
# List available record sets
print('Available Record Sets:')
record_sets = list(dataset.metadata.record_sets)
for rs in record_sets:
    print(f"- @id: {rs.id} | name: {rs.name}")

# For each record set, list its fields/columns by @id
for rs in record_sets:
    print(f"\nRecord Set: {rs.name} (@id: {rs.id})")
    columns = getattr(rs, 'columns', [])
    for col in columns:
        print(f"    Column: {getattr(col, 'name', '')} (@id: {col.id})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Select record sets by their @id
record_set_ids = [rs.id for rs in record_sets]

# Load all record sets to DataFrames
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

# Show available columns for the first record set
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"First record set loaded: {main_record_set_id}")
    print("Columns (by field @id):", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
We'll apply common data processing and exploration steps, such as filtering and normalization, using the DataFrame from the main record set.

In [ ]:
import numpy as np

df = dataframes[main_record_set_id].copy()

# List numeric columns for candidate selection
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print('Numeric Field @id candidates:', numeric_cols)

if numeric_cols:
    numeric_field = numeric_cols[0]  # For demonstration, pick the first
    print(f"Numeric field selected for analysis: {numeric_field}")

    # Filtering: Only keep rows where the numeric field value is above threshold
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Number of records with {numeric_field} > {threshold}: {len(filtered_df)}")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Find candidate group fields
    group_field_candidates = [col for col in df.columns if df[col].dtype == object]
    print("String Field (potential groupby keys):", group_field_candidates)
    group_field = None
    if group_field_candidates:
        group_field = group_field_candidates[0]
        grouped = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean of {numeric_field} by {group_field}:")
        display(grouped.head())
else:
    print('No numeric fields found in main record set. Please check the field types and @ids.')

## 5. Visualization
Let's visualize the distribution of the selected numeric field and, if available, its breakdown by a group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_cols:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field} (by @id)")
    plt.xlabel(numeric_field)
    plt.show()

    if group_field:
        plt.figure(figsize=(12, 6))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f"{numeric_field} by group field {group_field} (by @id)")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print('No numeric fields to visualize.')

## 6. Conclusion
- We've demonstrated how to load, inspect, and process the FAIR^2 dataset using the `mlcroissant` library.
- All dataset elements (record sets, columns) are referenced using their `@id` for reproducible workflows.
- Further analysis can extend this notebook to candidate fields and groupings specific to research goals.

Explore more about the Croissant standard and `mlcroissant` at [https://mlcommons.org/croissant/](https://mlcommons.org/croissant/).